# Setup

## Load packages

In [2]:
# Load up necessary packages. 
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Check GPU availability. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Import my custom utils.
import utils

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


## Load results

In [3]:
INPUT_DIR = "/tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs"

COMPARISON_TABLE = (
    "/tscc/nfs/home/kflanagan/projects/plip_plop/"
    "machine_learning_prototype/encode_initial_30_comparisons.tsv"
)

comparison_table = pd.read_csv(COMPARISON_TABLE, sep="\t")

comparison_table[["category", "experiment_A", "experiment_B"]].head()

,category,experiment_A,experiment_B
0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA
1,similar,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ
2,similar,PCBP1_HepG2_ENCSR256CHX,PCBP2_HepG2_ENCSR339FUY
3,similar,TIA1_HepG2_ENCSR623VEQ,TIAL1_HepG2_ENCSR322HHA
4,similar,SAFB_K562_ENCSR484LAB,SAFB2_K562_ENCSR943MHU


# Running pytorch

## Data setup

### Create metadata

In [4]:
all_signals = []
all_metadata = []

for comparison_id, row in comparison_table.iterrows():
    experiment_A = row["experiment_A"]
    experiment_B = row["experiment_B"]

    npz_file = os.path.join(
        INPUT_DIR,
        f"{experiment_A}_{experiment_B}.npz"
    )

    if not os.path.exists(npz_file):
        print(f"Missing: {npz_file}")
        continue

    data = np.load(npz_file)
    
    # Keep the original RPM signal for filtering and weighting.
    raw_signals = data["signals"].astype(np.float32)
    
    # Calculate signal strength before normalization.
    channel_totals = raw_signals.sum(axis=2)
    total_signal = channel_totals.sum(axis=1)
    
    # Log-transform the signal.
    signals = np.log1p(raw_signals)
    
    # Normalize each RBP independently by its L2 norm.
    channel_norms = np.sqrt((signals ** 2).sum(axis=2, keepdims=True))
    
    signals = np.divide(
        signals,
        channel_norms,
        out=np.zeros_like(signals),
        where=channel_norms > 0
    )
    
    n_windows = len(signals)

    all_signals.append(signals)

    metadata = pd.DataFrame({
        "comparison_id": comparison_id,
        "category": row["category"],
        "experiment_A": experiment_A,
        "experiment_B": experiment_B,
        "chrom": data["chrom"],
        "start": data["start"],
        "end": data["end"],
        "strand": data["strand"],
        "region_id": data["region_id"],
        "block_number": data["block_number"],
        "signal_A": channel_totals[:, 0],
        "signal_B": channel_totals[:, 1],
        "total_signal": total_signal
    })

    all_metadata.append(metadata)

In [5]:
signals = np.concatenate(all_signals, axis=0)
metadata = pd.concat(all_metadata, ignore_index=True)

print("Signal shape:", signals.shape)
print("Metadata shape:", metadata.shape)
print("Number of comparisons:", metadata["comparison_id"].nunique())

Signal shape: (717750, 2, 300)
Metadata shape: (717750, 13)
Number of comparisons: 30


#### Check comparisons

In [6]:
comparison_counts = (
    metadata
    .groupby(["comparison_id", "category", "experiment_A", "experiment_B"])
    .size()
    .reset_index(name="n_windows")
    .sort_values("n_windows", ascending=False)
)

comparison_counts

,comparison_id,category,experiment_A,experiment_B,n_windows
28,28,unrelated,SND1_HepG2_ENCSR061EVO,GRWD1_HepG2_ENCSR893NWB,114365
24,24,unrelated,KHSRP_K562_ENCSR438GZQ,NOLC1_K562_ENCSR001VAC,75892
5,5,similar,CSTF2_HepG2_ENCSR384MWO,CSTF2T_HepG2_ENCSR919HSE,54507
19,19,complex,PRPF4_HepG2_ENCSR977OXG,PRPF8_HepG2_ENCSR121NVA,53310
25,25,unrelated,YBX3_HepG2_ENCSR735HOK,DDX55_HepG2_ENCSR845VGB,47110
4,4,similar,SAFB_K562_ENCSR484LAB,SAFB2_K562_ENCSR943MHU,41498
18,18,complex,CPSF6_K562_ENCSR532VUB,CSTF2T_K562_ENCSR840DRD,33479
26,26,unrelated,ILF3_HepG2_ENCSR786TSC,NCBP2_HepG2_ENCSR018RVZ,29323
12,12,complex,PRPF8_HepG2_ENCSR121NVA,EFTUD2_HepG2_ENCSR527DXF,20344
23,23,unrelated,HNRNPC_HepG2_ENCSR550DVK,AKAP1_HepG2_ENCSR356ZMO,19439


In [7]:
print("Smallest comparison:", comparison_counts["n_windows"].min())
print("Largest comparison:", comparison_counts["n_windows"].max())
print("Ratio:", comparison_counts["n_windows"].max() / comparison_counts["n_windows"].min())

Smallest comparison: 1068
Largest comparison: 114365
Ratio: 107.08333333333333


### Weighting

In [8]:
# Set weighting alpha to 0.5 (square root). 
alpha = 0.5

# Create the weights. 
comparison_counts["comparison_weight"] = (
    comparison_counts["n_windows"] ** (alpha - 1)
)

# create a dictionary for quickly grabbing each weight for each comparison. 
weight_map = dict(zip(
    comparison_counts["comparison_id"],
    comparison_counts["comparison_weight"]
))

# Add weights to metadata. 
metadata["comparison_weight"] = metadata["comparison_id"].map(weight_map)

metadata["signal_weight"] = np.log1p(metadata["total_signal"])

metadata["signal_weight"] = (
    metadata["signal_weight"] /
    metadata.groupby("comparison_id")["signal_weight"].transform("mean")
)

metadata["weight"] = (
    metadata["comparison_weight"] *
    metadata["signal_weight"]
)

#### Check weights

In [9]:
effective_weights = (
    metadata
    .groupby("comparison_id")["weight"]
    .sum()
    .reset_index(name="total_weight")
)

effective_weights = effective_weights.merge(
    comparison_counts[
        ["comparison_id", "experiment_A", "experiment_B", "n_windows"]
    ],
    on="comparison_id"
)

effective_weights

,comparison_id,total_weight,experiment_A,experiment_B,n_windows
0,0,134.093256,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,17981
1,1,113.820036,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,12955
2,2,123.794185,PCBP1_HepG2_ENCSR256CHX,PCBP2_HepG2_ENCSR339FUY,15325
3,3,119.151163,TIA1_HepG2_ENCSR623VEQ,TIAL1_HepG2_ENCSR322HHA,14197
4,4,203.710575,SAFB_K562_ENCSR484LAB,SAFB2_K562_ENCSR943MHU,41498
5,5,233.467334,CSTF2_HepG2_ENCSR384MWO,CSTF2T_HepG2_ENCSR919HSE,54507
6,6,128.452331,IGF2BP1_K562_ENCSR975KIR,IGF2BP2_K562_ENCSR062NNB,16500
7,7,110.990989,FMR1_K562_ENCSR331VNX,FXR1_K562_ENCSR774RFN,12319
8,8,102.703455,SRSF1_K562_ENCSR432XUP,SRSF7_K562_ENCSR468FSW,10548
9,9,105.023805,DDX21_K562_ENCSR040QLV,DDX24_K562_ENCSR999WKT,11030


In [10]:
region_counts = (
    metadata
    .groupby("comparison_id")["region_id"]
    .nunique()
    .reset_index(name="n_regions")
)

comparison_counts = comparison_counts.merge(region_counts, on="comparison_id")

comparison_counts[
    ["category", "experiment_A", "experiment_B", "n_windows", "n_regions"]
].sort_values("n_windows")

,category,experiment_A,experiment_B,n_windows,n_regions
29,unrelated,FTO_K562_ENCSR989SMC,XRN2_K562_ENCSR657TZB,1068,818
28,complex,RPS3_K562_ENCSR120EAR,RPS11_K562_ENCSR269AJF,1331,997
27,complex,DKC1_HepG2_ENCSR301TFY,DDX55_HepG2_ENCSR845VGB,1719,1174
26,complex,LSM11_K562_ENCSR022BVV,SLBP_K562_ENCSR483NOP,2554,1656
25,complex,DROSHA_HepG2_ENCSR834YLD,DGCR8_HepG2_ENCSR061SZV,7995,4121
24,similar,SRSF1_K562_ENCSR432XUP,SRSF7_K562_ENCSR468FSW,10548,8068
23,similar,DDX21_K562_ENCSR040QLV,DDX24_K562_ENCSR999WKT,11030,8432
22,complex,SF3A3_HepG2_ENCSR331MIC,SF3B4_HepG2_ENCSR279UJF,11890,9321
21,similar,FMR1_K562_ENCSR331VNX,FXR1_K562_ENCSR774RFN,12319,8483
20,similar,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,12955,9297


## Train/val split setup. 

In [11]:
# Create a combined identifier from comparison and region. 
metadata["region_key"] = (
    metadata["comparison_id"].astype(str)
    + "_"
    + metadata["region_id"].astype(str)
)

In [12]:
# Setup random seed. 
rng = np.random.default_rng(42)

# Initialize masking vector. 
train_mask = np.zeros(len(metadata), dtype=bool)
val_mask = np.zeros(len(metadata), dtype=bool)

# Build mask. 
for comparison_id in metadata["comparison_id"].unique():
    comparison_rows = metadata["comparison_id"] == comparison_id

    comparison_regions = (
        metadata.loc[comparison_rows, "region_key"]
        .unique()
        .copy()
    )

    rng.shuffle(comparison_regions)

    split = int(len(comparison_regions) * 0.8)

    train_regions = comparison_regions[:split]
    val_regions = comparison_regions[split:]

    train_mask |= metadata["region_key"].isin(train_regions)
    val_mask |= metadata["region_key"].isin(val_regions)

/scratch/kflanagan/job_12070844/ipykernel_527312/2334664090.py:18: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(comparison_regions)


In [13]:
# Apply mask to the signals.  
train_signals = signals[train_mask]
val_signals = signals[val_mask]

# apply mask to the metadata. 
train_metadata = metadata.loc[train_mask].reset_index(drop=True)
val_metadata = metadata.loc[val_mask].reset_index(drop=True)

print("Training windows:", len(train_signals))
print("Validation windows:", len(val_signals))

Training windows: 574631
Validation windows: 143119


### Check train/val split

In [14]:
# Ensure no overlapping regions between the training and test dataset. 
overlap = set(train_metadata["region_key"]) & set(val_metadata["region_key"])

print("Overlapping regions:", len(overlap))

Overlapping regions: 0


In [15]:
# Check to ensure there are some training and validation windows for each comparison. 
split_counts = pd.DataFrame({
    "train": train_metadata.groupby("comparison_id").size(),
    "validation": val_metadata.groupby("comparison_id").size()
}).fillna(0).astype(int)

split_counts

,train,validation
comparison_id,,
0,14364,3617
1,10359,2596
2,12250,3075
3,11403,2794
4,33267,8231
5,43706,10801
6,13174,3326
7,9821,2498
8,8452,2096


In [16]:
# Check balance between categories for training and validation data. 
print("Training:")
print(train_metadata["category"].value_counts())
print("\nValidation:")
print(val_metadata["category"].value_counts())

Training:
category
unrelated    278559
similar      165605
complex      130467
Name: count, dtype: int64

Validation:
category
unrelated    69453
similar      41255
complex      32411
Name: count, dtype: int64


## Setup data loaders. 

In [17]:
# Create special data class for machine learning. 
class RBPWindowDataset(Dataset):
    def __init__(self, signals, weights):
        self.signals = torch.tensor(signals, dtype=torch.float32)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return(len(self.signals))

    def __getitem__(self, idx):
        return(self.signals[idx], self.weights[idx])

train_weights = train_metadata["weight"].to_numpy(dtype=np.float32)
val_weights = val_metadata["weight"].to_numpy(dtype=np.float32)

train_dataset = RBPWindowDataset(train_signals, train_weights)
val_dataset = RBPWindowDataset(val_signals, val_weights)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [18]:
batch_signals, batch_weights = next(iter(train_loader))

print("Signals:", batch_signals.shape)
print("Weights:", batch_weights.shape)
print("First few weights:", batch_weights[:10])

Signals: torch.Size([1024, 2, 300])
Weights: torch.Size([1024])
First few weights: tensor([0.0027, 0.0048, 0.0043, 0.0053, 0.0077, 0.0058, 0.0043, 0.0034, 0.0116,
        0.0033])


In [19]:
print("Train weight range:", train_weights.min(), train_weights.max())
print("Validation weight range:", val_weights.min(), val_weights.max())

Train weight range: 0.0016059256 0.06736074
Validation weight range: 0.0015899756 0.060359403


## Model setup

In [20]:
# Set model as autoencode
model = utils.SimpleAutoencoder(latent_dim=64).to(device)

# Select mean square error. 
criterion = nn.MSELoss()

# Adam optimizer (what is adam?)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Run model

In [21]:
train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    for batch_signals, batch_weights in train_loader:
        batch_signals = batch_signals.to(device)
        batch_weights = batch_weights.to(device)
    
        optimizer.zero_grad()
    
        reconstruction = model(batch_signals)
        loss = utils.weighted_mse_loss(
            reconstruction,
            batch_signals,
            batch_weights
        )
    
        loss.backward()
        optimizer.step()
    
        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for batch_signals, batch_weights in val_loader:
            batch_signals = batch_signals.to(device)
            batch_weights = batch_weights.to(device)
    
            reconstruction = model(batch_signals)
    
            loss = utils.weighted_mse_loss(
                reconstruction,
                batch_signals,
                batch_weights
            )
    
            total_val_loss += loss.item()

    average_val_loss = total_val_loss / len(val_loader)

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)

    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1}: "
        f"train = {average_train_loss:.6f}, "
        f"validation = {average_val_loss:.6f}"
    )

Epoch 1: train = 0.000944, validation = 0.000109
Epoch 2: train = 0.000079, validation = 0.000073
Epoch 3: train = 0.000059, validation = 0.000055
Epoch 4: train = 0.000051, validation = 0.000050
Epoch 5: train = 0.000047, validation = 0.000046
Epoch 6: train = 0.000044, validation = 0.000047
Epoch 7: train = 0.000042, validation = 0.000043
Epoch 8: train = 0.000041, validation = 0.000044
Epoch 9: train = 0.000040, validation = 0.000041
Epoch 10: train = 0.000040, validation = 0.000043
Epoch 11: train = 0.000039, validation = 0.000042
Epoch 12: train = 0.000039, validation = 0.000046
Epoch 13: train = 0.000038, validation = 0.000041
Epoch 14: train = 0.000038, validation = 0.000040
Epoch 15: train = 0.000038, validation = 0.000040
Epoch 16: train = 0.000038, validation = 0.000039
Epoch 17: train = 0.000037, validation = 0.000039
Epoch 18: train = 0.000037, validation = 0.000038
Epoch 19: train = 0.000037, validation = 0.000038
Epoch 20: train = 0.000036, validation = 0.000037


In [22]:
# Loads best model, not just last model. 
model.load_state_dict(best_model_state)
model = model.to(device)

# Save model for later use. 
torch.save(model.state_dict(), "/tscc/nfs/home/kflanagan/scratch/plip_plop_results/many_RBP_baseline.pt")

# Other stuff to save

In [63]:
val_loader

In [64]:
val_metadata

,comparison_id,category,experiment_A,experiment_B,chrom,start,end,strand,region_id,block_number,signal_A,signal_B,total_signal,comparison_weight,signal_weight,weight,region_key
0,0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,chr1,632418,632718,+,2,0,4107.650391,1979.252197,6086.902344,0.007457,1.535379,0.011450,0_2
1,0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,chr1,1035524,1035824,+,5,0,158.953857,99.873123,258.826965,0.007457,0.979650,0.007306,0_5
2,0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,chr1,1035824,1036124,+,5,1,234.992706,112.082657,347.075378,0.007457,1.031171,0.007690,0_5
3,0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,chr1,1036124,1036424,+,5,2,247.813721,144.584747,392.398468,0.007457,1.052738,0.007851,0_5
4,0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,chr1,1036424,1036724,+,5,3,173.799271,105.103882,278.903137,0.007457,0.992764,0.007404,0_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143114,29,unrelated,QKI_K562_ENCSR366YOG,NOLC1_K562_ENCSR001VAC,chrX,154712409,154712709,-,10563,0,7.533993,96.883583,104.417580,0.007705,0.861073,0.006634,29_10563
143115,29,unrelated,QKI_K562_ENCSR366YOG,NOLC1_K562_ENCSR001VAC,chrX,154781550,154781850,-,10565,0,33.763027,262.064392,295.827423,0.007705,1.052446,0.008109,29_10565
143116,29,unrelated,QKI_K562_ENCSR366YOG,NOLC1_K562_ENCSR001VAC,chrX,154783317,154783617,-,10566,0,37.535416,188.196411,225.731827,0.007705,1.002647,0.007725,29_10566
143117,29,unrelated,QKI_K562_ENCSR366YOG,NOLC1_K562_ENCSR001VAC,chrX,154792041,154792341,-,10568,0,23.011919,181.809570,204.821487,0.007705,0.984760,0.007587,29_10568
